# Appendix B: Fourier Coefficients of the Truncated Cosine

This notebook derives the Fourier coefficients $c_n$ of the visibility function
$\Pi(t) = \max\{\cos\beta(t),\, 0\}$ and verifies the results in Appendix `app:fourier_visibility` (Appendix B) and Section
`subsec:ft_cosine` of the paper. Notation: the notebook's $\Pi(t)$, $a_0$, $a_1$ are the paper's
$\mathcal{V}(t)$, $b_0$, $b_1$.

**Key results:**
- The defining integral (Eq. `cn_integral`), split into the two integrals of Eq. `cn_split` (Eqs. `cn_int_const`, `cn_int_cos`)
- General $c_n$ formula (Eqs. `cn_result`, `cn_general`)
- Special cases $c_0$, $c_{\pm 1}$ (stated after Eq. `cn_general`) as removable singularities
- Edge-on ($I=90°$) coefficients (Eq. `cn_edgeon`)
- Visibility window and inclination form (Eqs. `theta_vis`, `theta_vis_inc`, `cn_inc`), pole-on limit
- Squared coefficients $|c_n|^2$ for the kernel (Eqs. `kernel_n2`, `kernel_edgeon_closed`)

The paper's equations are taken from the equation registry (`equations.py`), where each is transcribed
into SymPy once, exactly as typeset; `check_equations.py` keeps the transcriptions in sync with `ms.tex`.
Every derived result is compared with them.

In [1]:
import sympy as sp
from sympy import pi, cos, sin, integrate, simplify, symbols, latex, exp, I as im
from IPython.display import display, Math
sp.init_printing()

## Setup

The visibility function $\Pi(t) = \max\{\cos\beta(t),\, 0\}$ is periodic with period $P = 2\pi/\omega_0$.
With $\cos\beta(t) = a_0 + a_1\cos(\omega_0 t + \Lambda_0)$ where $a_0 = \cos I\sin\Phi$, $a_1 = \sin I\cos\Phi$,
the spot is visible when $|\omega_0 t + \Lambda_0| < \theta_{\rm vis}$, where
$\theta_{\rm vis} = \arccos(-a_0/a_1)$.

The Fourier coefficients are:
$$c_n = \frac{1}{2\pi}\int_{-\theta_{\rm vis}}^{\theta_{\rm vis}} (a_0 + a_1\cos\theta)\, e^{-in\theta}\, d\theta$$

## Evaluating the integral

Eq. `cn_split` splits the integral of Eq. `cn_integral` into constant and cosine parts:
$$c_n = \frac{1}{2\pi}\left[a_0\int_{-\theta_{\rm vis}}^{\theta_{\rm vis}} e^{-in\theta}\,d\theta + a_1\int_{-\theta_{\rm vis}}^{\theta_{\rm vis}} \cos\theta\; e^{-in\theta}\,d\theta\right]$$
Both are taken from the registry. Check that the split is the same integral, then evaluate its two
integrals as typeset.

In [2]:
import equations as eqs   # the paper's equations, transcribed once (src/derivations/equations.py)

theta = sp.Symbol('theta', real=True)
n = sp.Symbol('n', integer=True, nonzero=True)
tv = sp.Symbol('theta_v', positive=True)  # theta_vis
a0 = sp.Symbol('a_0', real=True)
a1 = sp.Symbol('a_1', positive=True)
# Registry (paper) symbols -> this notebook's: b_0 -> a_0, b_1 -> a_1, theta_vis -> theta_v, and n != 0 here
to_nb = {eqs.b0: a0, eqs.b1: a1, eqs.theta_vis: tv, eqs.n: n}

# Eqs. cn_integral and cn_split as typeset
cn_integral_paper = eqs.get("eq:cn_integral").expr.subs(to_nb)
cn_split_paper = sp.expand(eqs.get("eq:cn_split").expr.subs(to_nb))
Int1 = 2 * pi * cn_split_paper.coeff(a0)   # constant part: int e^{-in theta} d theta
Int2 = 2 * pi * cn_split_paper.coeff(a1)   # cosine part: int cos(theta) e^{-in theta} d theta

# Eq. cn_split is Eq. cn_integral with the integrand split: same limits, integrand and 1/(2 pi)
(Int_full,) = cn_integral_paper.atoms(sp.Integral)
assert Int_full.limits == Int1.limits == Int2.limits == ((theta, -tv, tv),)
assert simplify(sp.expand(Int_full.function - (a0 * Int1.function + a1 * Int2.function))) == 0
assert simplify(cn_integral_paper / Int_full - 1 / (2 * pi)) == 0
print("Verified: Eq. cn_split is Eq. cn_integral split into its constant and cosine parts")

# First integral (constant part) of Eq. cn_split
int1 = Int1.doit()
int1_simplified = simplify(int1)
print("First integral (n != 0):")
display(Math(rf"\int_{{-\theta_v}}^{{\theta_v}} e^{{-in\theta}}\,d\theta = {latex(int1_simplified)}"))

Verified: Eq. cn_split is Eq. cn_integral split into its constant and cosine parts
First integral (n != 0):


<IPython.core.display.Math object>

In [3]:
# Second integral (cosine part) of Eq. cn_split
# = (1/2) int e^{i(1-n)theta} + (1/2) int e^{-i(1+n)theta}
# For n != +/- 1, each gives 2*sin(m*tv)/m

int2 = Int2.doit()
int2_simplified = simplify(int2)
print("Second integral (n != 0, +/-1):")
display(Math(rf"\int_{{-\theta_v}}^{{\theta_v}} \cos\theta\, e^{{-in\theta}}\,d\theta = {latex(int2_simplified)}"))

Second integral (n != 0, +/-1):


<IPython.core.display.Math object>

In [4]:
# General c_n for n != 0, +/-1
cn_general = (a0 * int1_simplified + a1 * int2_simplified) / (2 * pi)
cn_general = simplify(cn_general)
print("General c_n (Eq. cn_result):")
display(Math(rf"c_n = {latex(cn_general)}"))

General c_n (Eq. cn_result):


<IPython.core.display.Math object>

### Check against the paper: Eqs. `cn_int_const`, `cn_int_cos`, `cn_result` and `cn_general`

The appendix prints the two integrals of Eq. `cn_split`, $2\sin(n\theta_{\rm vis})/n$ (Eq. `cn_int_const`) and
$\frac{\sin((1-n)\theta_{\rm vis})}{1-n} + \frac{\sin((1+n)\theta_{\rm vis})}{1+n}$ (Eq. `cn_int_cos`), and the general coefficient
Eq. `cn_result`, whose right-hand side is repeated in the boxed main-text Eq. `cn_general`. They are taken
from the registry with $b_0 \to a_0$, $b_1 \to a_1$. SymPy returns the results above as a `Piecewise`
whose `otherwise` branch is the generic case $n \neq \pm 1$; that branch is compared symbolically in $n$,
and the full result is also compared at explicit integers $2 \le |n| \le 8$.

In [5]:
def is_zero(expr):
    # Rewrite trig functions as complex exponentials before simplifying (the derived results mix both).
    return simplify(sp.expand(expr.rewrite(exp))) == 0


def generic_branch(expr):
    # Branch of a Piecewise that holds for generic n (condition True); the expression itself otherwise.
    if isinstance(expr, sp.Piecewise):
        branches = [e for e, cond in expr.args if cond == True]
        assert len(branches) == 1
        return branches[0]
    return expr


b0, b1 = a0, a1   # paper notation

paper_int1 = eqs.get("eq:cn_int_const").expr.subs(to_nb)       # first integral, n != 0
paper_int2 = eqs.get("eq:cn_int_cos").expr.subs(to_nb)         # second integral, n != +/-1
paper_cn = eqs.get("eq:cn_result").expr.subs(to_nb)            # n != 0, +/-1
paper_cn_general = eqs.get("eq:cn_general").expr.subs(to_nb)   # its closed-form member

assert is_zero(generic_branch(int1_simplified) - paper_int1), "first integral != Eq. cn_int_const"
assert is_zero(generic_branch(int2_simplified) - paper_int2), "second integral != Eq. cn_int_cos"
assert is_zero(generic_branch(cn_general) - paper_cn), "general c_n != Eq. cn_result"
assert is_zero(generic_branch(cn_general) - paper_cn_general), "general c_n != Eq. cn_general"
for nn in [k for k in range(-8, 9) if abs(k) >= 2]:
    assert is_zero(cn_general.subs(n, nn) - paper_cn.subs(n, nn)), f"c_{nn} != Eq. cn_general"
display(Math(rf"c_n^{{\rm paper}} = {latex(paper_cn)}"))
print("Verified: Eqs. cn_int_const, cn_int_cos, cn_result and cn_general (symbolic n, and 2 <= |n| <= 8)")

<IPython.core.display.Math object>

Verified: Eqs. cn_int_const, cn_int_cos, cn_result and cn_general (symbolic n, and 2 <= |n| <= 8)


## Special cases

In [6]:
# c_0: n=0, integral of (a0 + a1*cos(theta)) from -tv to tv
c0_expr = integrate(a0 + a1 * cos(theta), (theta, -tv, tv)) / (2 * pi)
c0_expr = simplify(c0_expr)
print("c_0 (text after Eq. cn_general):")
display(Math(rf"c_0 = {latex(c0_expr)}"))

# Verify: should be (a0*tv + a1*sin(tv)) / pi
expected_c0 = (a0 * tv + a1 * sin(tv)) / pi
assert simplify(c0_expr - expected_c0) == 0, "c_0 mismatch!"
print("  Verified: c_0 = (a_0 * theta_vis + a_1 * sin(theta_vis)) / pi")

c_0 (text after Eq. cn_general):


<IPython.core.display.Math object>

  Verified: c_0 = (a_0 * theta_vis + a_1 * sin(theta_vis)) / pi


In [7]:
# c_1: n=1, the (n-1) term is singular; integrate directly
c1_integrand = (a0 + a1 * cos(theta)) * exp(-im * theta)
c1_expr = integrate(c1_integrand, (theta, -tv, tv)) / (2 * pi)
c1_expr = simplify(sp.re(c1_expr))  # c_1 is real for real Pi(t)
print("c_1 (text after Eq. cn_general):")
display(Math(rf"c_1 = {latex(c1_expr)}"))

# Verify: should be (1/pi) * (a0*sin(tv) + a1/2*(tv + sin(2*tv)/2))
expected_c1 = (a0 * sin(tv) + a1 / 2 * (tv + sin(2 * tv) / 2)) / pi
assert simplify(c1_expr - expected_c1) == 0, "c_1 mismatch!"
print("  Verified: c_1 = (a_0*sin(tv) + a_1/2*(tv + sin(2*tv)/2)) / pi")

c_1 (text after Eq. cn_general):


<IPython.core.display.Math object>

  Verified: c_1 = (a_0*sin(tv) + a_1/2*(tv + sin(2*tv)/2)) / pi


### Check against the paper: $c_0$, $c_{\pm 1}$ and the removable singularities (text after Eq. `cn_general`)

The paper prints $c_0 = (b_0\theta_{\rm vis} + b_1\sin\theta_{\rm vis})/\pi$ and
$c_{\pm 1} = (b_0\sin\theta_{\rm vis} + b_1[\theta_{\rm vis} + \sin\theta_{\rm vis}\cos\theta_{\rm vis}]/2)/\pi$, and states
that these are removable singularities of Eq. `cn_general`. Checked: (i) the derived $c_0$, $c_1$ equal the
printed forms; (ii) $c_1$ before taking the real part above, and $c_{-1}$ by direct integration, both equal
the printed $c_{\pm 1}$ (so no imaginary part was discarded); (iii) the $n = \pm 1$ branch of the general
`Piecewise` result agrees; (iv) the limits $n \to 0$ and $n \to \pm 1$ of Eq. `cn_general`, with $n$ treated as
continuous, reproduce $c_0$ and $c_{\pm 1}$.

In [8]:
# Inline in the text after Eq. cn_general, so transcribed here rather than in the registry
paper_c0 = (b0 * tv + b1 * sin(tv)) / pi
paper_c1 = (b0 * sin(tv) + b1 * (tv + sin(tv) * cos(tv)) / 2) / pi   # c_{+1} = c_{-1}

assert is_zero(c0_expr - paper_c0), "c_0 != paper"
assert is_zero(c1_expr - paper_c1), "c_1 != paper"

c1_full = integrate(c1_integrand, (theta, -tv, tv)) / (2 * pi)   # before taking re()
assert is_zero(c1_full - paper_c1), "c_1 has a nonzero imaginary part"
cm1_expr = integrate((a0 + a1 * cos(theta)) * exp(im * theta), (theta, -tv, tv)) / (2 * pi)
assert is_zero(cm1_expr - paper_c1), "c_{-1} != paper"
for nn in (1, -1):
    assert is_zero(cn_general.subs(n, nn) - paper_c1), f"n = {nn} branch of the general result != paper"

nu = sp.Symbol('nu', real=True)   # continuous harmonic index
paper_cn_nu = paper_cn.subs(n, nu)
assert is_zero(sp.limit(paper_cn_nu, nu, 0) - paper_c0), "n -> 0 limit of Eq. cn_general != c_0"
assert is_zero(sp.limit(paper_cn_nu, nu, 1) - paper_c1), "n -> 1 limit of Eq. cn_general != c_1"
assert is_zero(sp.limit(paper_cn_nu, nu, -1) - paper_c1), "n -> -1 limit of Eq. cn_general != c_-1"
print("Verified: c_0 and c_{+/-1} as printed, real, and equal to the n -> 0, +/-1 limits of Eq. cn_general")

Verified: c_0 and c_{+/-1} as printed, real, and equal to the n -> 0, +/-1 limits of Eq. cn_general


## Edge-on verification ($I = 90°$)

For $I = \pi/2$: $a_0 = 0$, $a_1 = \cos\Phi$, $\theta_{\rm vis} = \pi/2$.
All $c_n$ reduce to $c_n = g_n \cos\Phi$.

In [9]:
Phi = sp.Symbol('Phi', positive=True)

# Substitutions for I = pi/2
edge_subs = {a0: 0, a1: cos(Phi), tv: pi / 2}

# c_0 at I=90
c0_90 = simplify(c0_expr.subs(edge_subs))
display(Math(rf"c_0 = {latex(c0_90)}"))

# c_1 at I=90
c1_90 = simplify(c1_expr.subs(edge_subs))
display(Math(rf"c_1 = {latex(c1_90)}"))

# c_2: use general formula with n=2
cn_at_2 = cn_general.subs(n, 2)
c2_90 = simplify(cn_at_2.subs(edge_subs))
display(Math(rf"c_2 = {latex(c2_90)}"))

# Verify expected values
assert simplify(c0_90 - cos(Phi) / pi) == 0
assert simplify(c1_90 - cos(Phi) / 4) == 0
assert simplify(c2_90 - cos(Phi) / (3 * pi)) == 0
print("All edge-on values verified.")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

All edge-on values verified.


In [10]:
# Pattern for even harmonics at I=90:
# c_n = (-1)^(n/2+1) * cos(Phi) / (pi*(n^2-1))  for even n >= 2
print("Even harmonic pattern at I=90°:")
for nn in [2, 4, 6, 8]:
    cn_val = simplify(cn_general.subs(n, nn).subs(edge_subs))
    expected = (-1)**(nn//2 + 1) * cos(Phi) / (pi * (nn**2 - 1))
    match = simplify(cn_val - expected) == 0
    assert match, f"c_{nn} does not follow the even-harmonic pattern"
    display(Math(rf"c_{{{nn}}} = {latex(cn_val)} \quad \checkmark" if match else rf"c_{{{nn}}} = {latex(cn_val)} \quad \times"))

# Odd harmonics n >= 3 vanish
print("\nOdd harmonics (n >= 3) vanish:")
for nn in [3, 5, 7]:
    cn_val = simplify(cn_general.subs(n, nn).subs(edge_subs))
    assert cn_val == 0, f"c_{nn} should vanish at I = 90 deg"
    display(Math(rf"c_{{{nn}}} = {latex(cn_val)}"))

Even harmonic pattern at I=90°:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


Odd harmonics (n >= 3) vanish:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Check against the paper: edge-on coefficients (Eq. `cn_edgeon`)

Eq. `cn_edgeon`, a function of $n$ in the registry, is compared with the derived coefficients for
$|n| \le 10$, including negative $n$ (the paper states $c_{-n} = c_n$, all real).

In [11]:
def paper_cn_edgeon(k):
    # Eq. cn_edgeon at n = k
    return eqs.get("eq:cn_edgeon").expr.subs({eqs.n: k, eqs.phi: Phi})


derived_edgeon = {0: c0_90, 1: c1_90, -1: simplify(cm1_expr.subs(edge_subs))}
for k in range(-10, 11):
    if abs(k) >= 2:
        derived_edgeon[k] = simplify(cn_general.subs(n, k).subs(edge_subs))
    assert simplify(derived_edgeon[k] - paper_cn_edgeon(k)) == 0, f"c_{k} != Eq. cn_edgeon"
    assert simplify(sp.im(derived_edgeon[k])) == 0, f"c_{k} is not real"
for k in range(1, 11):
    assert simplify(derived_edgeon[-k] - derived_edgeon[k]) == 0, f"c_-{k} != c_{k}"
print("Verified: Eq. cn_edgeon for |n| <= 10; all coefficients real with c_{-n} = c_n")

Verified: Eq. cn_edgeon for |n| <= 10; all coefficients real with c_{-n} = c_n


### Check against the paper: visibility window and inclination form (Eqs. `theta_vis`, `theta_vis_inc`, `cn_inc`)

- Eq. `theta_vis`: $\cos\beta = b_0 + b_1\cos\theta$ vanishes at $\theta_{\rm vis} = \arccos(-b_0/b_1)$.
- Eq. `theta_vis_inc` is Eq. `theta_vis` with $b_0 = \cos I\sin\Phi$, $b_1 = \sin I\cos\Phi$, so that $-b_0/b_1 = -\cot I\tan\Phi$; the window is
  $\theta_{\rm vis} = \pi$ at $\Phi = I$ and $0$ at $\Phi = -I$ (the partition described after Eq. `cn_inc`).
- Eq. `cn_inc` is Eq. `cn_general` under that substitution.
- Edge-on ($I = \pi/2$): $b_0 = 0$, $b_1 = \cos\Phi$, $\theta_{\rm vis} = \pi/2$, i.e. the substitution `edge_subs` used above.
- Circumpolar ($\theta_{\rm vis} = \pi$): only $c_0 = b_0$ and $c_{\pm1} = b_1/2$ survive. Pole-on ($I \to 0$): the visibility
  is the constant $\sin\Phi$ on the northern hemisphere, all $c_{n\ge1} \to 0$, and
  $\langle|c_0|^2\rangle_\Phi = 1/4$ for $p(\Phi) = 1/\pi$, which in Eq. `kernel_n2` gives the stated limit
  $\mathcal{K}(\tau) \to \sigma_k^2 R_\Gamma(\tau)/4$.

In [12]:
Inc, PhiL = eqs.inc, eqs.phi   # stellar inclination, and spot latitude on [-pi/2, pi/2]
b0_inc, b1_inc = cos(Inc) * sin(PhiL), sin(Inc) * cos(PhiL)

# Eq. theta_vis: cos(beta) = b0 + b1 cos(theta) vanishes at theta_vis
paper_theta_vis = eqs.get("eq:theta_vis").expr.subs(to_nb)
assert simplify(b0 + b1 * cos(paper_theta_vis)) == 0

# Eq. theta_vis_inc: Eq. theta_vis with b0 = cos I sin Phi, b1 = sin I cos Phi
paper_theta_vis_inc = eqs.get("eq:theta_vis_inc").expr
assert simplify(paper_theta_vis.subs({b0: b0_inc, b1: b1_inc}).args[0] - paper_theta_vis_inc.args[0]) == 0
assert simplify(paper_theta_vis_inc.subs(PhiL, Inc)) == pi
assert simplify(paper_theta_vis_inc.subs(PhiL, -Inc)) == 0

# Eq. cn_inc: Eq. cn_general with the same substitution
paper_cn_inc = eqs.get("eq:cn_inc").expr.subs(to_nb)
assert simplify(paper_cn_inc - paper_cn_general.subs({a0: b0_inc, a1: b1_inc})) == 0, "Eq. cn_inc != Eq. cn_general"

# Edge-on: I = pi/2 reproduces edge_subs
assert b0_inc.subs(Inc, pi / 2) == 0 and b1_inc.subs(Inc, pi / 2) == cos(PhiL)
assert paper_theta_vis_inc.subs(Inc, pi / 2) == pi / 2

# Circumpolar window theta_vis = pi
assert simplify(paper_c0.subs(tv, pi) - b0) == 0
assert simplify(paper_c1.subs(tv, pi) - b1 / 2) == 0
for nn in [k for k in range(-8, 9) if abs(k) >= 2]:
    assert simplify(paper_cn.subs(n, nn).subs(tv, pi)) == 0
# Never-visible window theta_vis = 0
assert paper_c0.subs(tv, 0) == 0 and simplify(paper_c1.subs(tv, 0)) == 0

# Pole-on: circumpolar northern hemisphere with b0 = sin(Phi), b1 = 0; hidden southern hemisphere
c0_pole = paper_c0.subs(tv, pi).subs(a0, b0_inc).subs(Inc, 0)
c1_pole = paper_c1.subs(tv, pi).subs(a1, b1_inc).subs(Inc, 0)
assert c0_pole == sin(PhiL) and c1_pole == 0
c0sq_avg_pole = integrate(c0_pole**2 / pi, (PhiL, 0, pi / 2))   # southern hemisphere contributes 0
assert c0sq_avg_pole == sp.Rational(1, 4), "pole-on <|c_0|^2> != 1/4"
# ... and in Eq. kernel_n2 (c_1 = c_2 = 0) this is the stated limit K -> sigma_k^2 R_Gamma(tau) / 4
K_n2 = eqs.get("eq:kernel_n2").expr
K_pole = K_n2.subs({eqs.C0: c0sq_avg_pole, eqs.C1: 0, eqs.C2: 0})
assert simplify(K_pole - eqs.sigma_k**2 * eqs.R_Gamma(eqs.tau) / 4) == 0, "pole-on limit of Eq. kernel_n2"
print("Verified: Eqs. theta_vis, theta_vis_inc, cn_inc; edge-on substitution; circumpolar and pole-on limits")

Verified: Eqs. theta_vis, theta_vis_inc, cn_inc; edge-on substitution; circumpolar and pole-on limits


## Squared coefficients for the kernel

The GP kernel involves $\langle|c_n|^2\rangle_\Phi = \frac{1}{\pi}\int_0^\pi |c_n(\Phi)|^2\,d\Phi$.

At $I=90°$, $c_n = g_n\cos\Phi$ and $\int_0^\pi \cos^2\Phi\,d\Phi = \pi/2$, giving $\langle|c_n|^2\rangle = g_n^2/2$.

In [13]:
print("Latitude-averaged |c_n|^2 at I=90°, uniform p(Phi) on [0, pi]:")
print("=" * 60)

cn_values = {0: c0_90, 1: c1_90, 2: c2_90}
cn_sq_avg = {}

for nn in [0, 1, 2]:
    cn = cn_values[nn]
    result = simplify(integrate(cn**2, (Phi, 0, pi)) / pi)
    cn_sq_avg[nn] = result
    display(Math(rf"\langle|c_{nn}|^2\rangle = {latex(result)}"))

# Verify g_n^2 / 2 formula
g_vals = {0: 1/pi, 1: sp.Rational(1, 4), 2: 1/(3*pi)}
for nn in [0, 1, 2]:
    expected = simplify(g_vals[nn]**2 / 2)
    assert simplify(cn_sq_avg[nn] - expected) == 0, f"Mismatch for n={nn}"
print("\nAll verified: <|c_n|^2> = g_n^2 / 2")

Latitude-averaged |c_n|^2 at I=90°, uniform p(Phi) on [0, pi]:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>


All verified: <|c_n|^2> = g_n^2 / 2


### Check against the paper: latitude averages and the edge-on kernel (Eqs. `kernel_n2`, `kernel_edgeon_closed`)

The paper averages over $p(\Phi) = 1/\pi$ on $[-\pi/2, \pi/2]$ (the cell above integrates over $[0, \pi]$, which is
equivalent for $\cos^2\Phi$). With the paper's interval, Eq. `kernel_n2` with these latitude averages,
$\sigma_k^2 R_\Gamma(\tau)\left[\langle|c_0|^2\rangle + 2\langle|c_1|^2\rangle\cos(\omega_0\tau) + 2\langle|c_2|^2\rangle\cos(2\omega_0\tau)\right]$,
must equal Eq. `kernel_edgeon_closed`, whose bracket is
$\frac{1}{2\pi^2} + \frac{1}{16}\cos(\omega_0\tau) + \frac{1}{9\pi^2}\cos(2\omega_0\tau)$.
The $g_n$ used above are the paper's $g_0 = 1/\pi$, $g_1 = 1/4$, $g_2 = 1/(3\pi)$.

In [14]:
w0, lag = eqs.omega0, eqs.tau
cn_sq_avg_paper = {nn: integrate(cn_values[nn].subs(Phi, PhiL)**2 / pi, (PhiL, -pi / 2, pi / 2)) for nn in [0, 1, 2]}
for nn in [0, 1, 2]:
    assert simplify(cn_sq_avg_paper[nn] - g_vals[nn]**2 / 2) == 0
bracket_derived = (cn_sq_avg_paper[0] + 2 * cn_sq_avg_paper[1] * cos(w0 * lag)
                   + 2 * cn_sq_avg_paper[2] * cos(2 * w0 * lag))

K_edgeon_paper = eqs.get("eq:kernel_edgeon_closed").expr
K_derived = K_n2.subs({eqs.C0: cn_sq_avg_paper[0], eqs.C1: cn_sq_avg_paper[1], eqs.C2: cn_sq_avg_paper[2]})
assert simplify(K_derived - K_edgeon_paper) == 0, "Eq. kernel_n2 with <|c_n|^2> = g_n^2/2 != Eq. kernel_edgeon_closed"
assert simplify(K_derived - eqs.sigma_k**2 * eqs.R_Gamma(lag) * bracket_derived) == 0
display(Math(rf"\left[\ldots\right] = {latex(bracket_derived)}"))
print("Verified: <|c_n|^2> = g_n^2/2 on [-pi/2, pi/2], and Eq. kernel_n2 with them is Eq. kernel_edgeon_closed")

<IPython.core.display.Math object>

Verified: <|c_n|^2> = g_n^2/2 on [-pi/2, pi/2], and Eq. kernel_n2 with them is Eq. kernel_edgeon_closed


In [15]:
# Power budget: fraction of total power in each harmonic
import numpy as np

total_power = float(cn_sq_avg[0]) + 2 * sum(float(cn_sq_avg[nn]) for nn in [1, 2])
print("Power budget at I=90°:")
print(f"  n=0 (DC):          {float(cn_sq_avg[0])/total_power:.1%}")
print(f"  n=+/-1 (fund.):    {2*float(cn_sq_avg[1])/total_power:.1%}")
print(f"  n=+/-2 (2nd harm): {2*float(cn_sq_avg[2])/total_power:.1%}")
print(f"  Total (n<=2):      {total_power:.6f}")
print(f"\nThe second harmonic contributes {2*float(cn_sq_avg[2])/total_power:.0%} of the total power,")
print("justifying truncation to n<=2 in the main text.")

Power budget at I=90°:
  n=0 (DC):          40.7%
  n=+/-1 (fund.):    50.2%
  n=+/-2 (2nd harm): 9.0%
  Total (n<=2):      0.124419

The second harmonic contributes 9% of the total power,
justifying truncation to n<=2 in the main text.


### Check against the paper: share of variance at $n = 2$ for $I = 90°$ (Section `subsec:harmonic_n2`)

The paper states that the $n = 2$ share of the variance is "$9\%$ of the total at best (equator-on)". At
$I = 90°$ every $c_n \propto \cos\Phi$, so the share $2\langle|c_2|^2\rangle / \sum_n\langle|c_n|^2\rangle$ does not depend on
$p(\Phi)$. It is evaluated both with the $n \le 2$ truncation of the cell above and with all harmonics, using
Parseval, $\sum_n |c_n|^2 = \frac{1}{2\pi}\int_{-\pi}^{\pi}\mathcal{V}^2\,d\theta$, which gives $\langle\sum_n|c_n|^2\rangle_\Phi = 1/8$.

In [16]:
total_all = integrate(integrate((cos(PhiL) * cos(theta))**2, (theta, -pi / 2, pi / 2)) / (2 * pi) / pi,
                      (PhiL, -pi / 2, pi / 2))
assert total_all == sp.Rational(1, 8)
share2_all = float(2 * cn_sq_avg[2] / total_all)
share2_trunc = 2 * float(cn_sq_avg[2]) / total_power
print(f"n = 2 share at I = 90 deg: {share2_trunc:.2%} (n <= 2), {share2_all:.2%} (all harmonics)")
assert round(100 * share2_trunc) == 9 and round(100 * share2_all) == 9, "n = 2 share != 9% (Section subsec:harmonic_n2)"
print("Verified: the n = 2 harmonic carries 9% of the variance at I = 90 deg")

n = 2 share at I = 90 deg: 9.05% (n <= 2), 9.01% (all harmonics)
Verified: the n = 2 harmonic carries 9% of the variance at I = 90 deg
